### baseline model rank

In [ ]:
import os
import re
import json
import random
import glob
import itertools
import time

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import Path 

PROJECT_ROOT = Path.cwd()

COLAB_IMPORTS_DIR = os.path.join(PROJECT_ROOT, "colab_imports")
COLAB_EXPORTS_DIR = os.path.join(PROJECT_ROOT, "colab_exports")

RUN_PREFIXES = {
    "CrossEncoder": "cross_encoder_full100_",
    "GPT": "gpt_top5_",
    "Llama": "llama100_top5_",
    "Qwen": "qwen100_top5_",
}

METHOD_ORDER = ["BM25", "MMR", "CrossEncoder", "GPT", "Qwen", "Llama", "Random"]

TARGET_REL = 2
RANDOM_SEED = 2026
MMR_LAMBDA = 0.7
BASELINE_PRINT_EVERY = 10

STOPWORDS = set(ENGLISH_STOP_WORDS)

def norm_ws(s):
    return re.sub(r"\s+", " ", (s or "")).strip()

def tok_content(s):
    toks = re.findall(r"[a-z0-9]+", (s or "").lower())
    out = []
    for t in toks:
        if t in STOPWORDS:
            continue
        if t.isdigit():
            continue
        out.append(t)
    return out

def jaccard_set(a, b):
    a = set(a)
    b = set(b)
    if not a and not b:
        return 1.0
    return len(a & b) / max(1, len(a | b))

def overlap_count(a, b, k):
    return len(set(a[:k]) & set(b[:k]))

def exact_topk_list_match(a, b, k):
    return int(list(a[:k]) == list(b[:k]))

def exact_topk_set_match(a, b, k):
    return int(set(a[:k]) == set(b[:k]))

def precision_at_k(labels, ranked_indices, k, rel_threshold):
    top = ranked_indices[:k]
    if len(top) < k:
        return None
    good = sum(1 for ix in top if labels[ix] >= rel_threshold)
    return good / k

def recall_at_k(labels, ranked_indices, k, rel_threshold):
    top = ranked_indices[:k]
    if len(top) < k:
        return None
    total_good = sum(1 for x in labels if x >= rel_threshold)
    if total_good == 0:
        return 0.0
    good = sum(1 for ix in top if labels[ix] >= rel_threshold)
    return good / total_good

def hit_at_k(labels, ranked_indices, k, rel_threshold):
    top = ranked_indices[:k]
    if len(top) < k:
        return None
    return float(any(labels[ix] >= rel_threshold for ix in top))

def dcg_at_k(labels, ranked_indices, k):
    top = ranked_indices[:k]
    if len(top) < k:
        return None
    vals = []
    for rank_pos, ix in enumerate(top, start=1):
        rel = labels[ix]
        gain = (2 ** max(rel, 0) - 1) / np.log2(rank_pos + 1)
        vals.append(gain)
    return float(sum(vals))

def ndcg_at_k(labels, ranked_indices, k):
    actual = dcg_at_k(labels, ranked_indices, k)
    if actual is None:
        return None
    ideal_order = sorted(range(len(labels)), key=lambda i: labels[i], reverse=True)
    ideal = dcg_at_k(labels, ideal_order, k)
    if ideal == 0:
        return 0.0
    return actual / ideal

def bm25_rank_within(query, candidates):
    corpus = [tok_content(d["passage_text"]) for d in candidates]
    bm25 = BM25Okapi(corpus)
    scores = bm25.get_scores(tok_content(query))
    order = sorted(range(len(scores)), key=lambda x: scores[x], reverse=True)
    return order, scores.tolist()

def jaccard_text_content(a, b):
    a = set(tok_content(a))
    b = set(tok_content(b))
    return len(a & b) / max(1, len(a | b)) if (a or b) else 0.0

def mmr_rank_within_bm25rel(query, candidates, lam):
    corpus = [tok_content(d["passage_text"]) for d in candidates]
    bm25 = BM25Okapi(corpus)
    rel_scores = bm25.get_scores(tok_content(query))

    selected = []
    remaining = list(range(len(candidates)))

    while remaining:
        best_ix = None
        best_score = None

        for ix in remaining:
            r = float(rel_scores[ix])
            div = 0.0
            if selected:
                div = max(
                    jaccard_text_content(candidates[ix]["passage_text"], candidates[j]["passage_text"])
                    for j in selected
                )
            score = lam * r - (1.0 - lam) * div
            if best_score is None or score > best_score:
                best_score = score
                best_ix = ix

        selected.append(best_ix)
        remaining.remove(best_ix)

    return selected

def random_rank_within(example_idx, candidates_len, seed):
    rng = random.Random(seed + int(example_idx))
    order = list(range(candidates_len))
    rng.shuffle(order)
    return order

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def find_latest_run_dir(base_dir, prefix):
    candidates = []
    for name in os.listdir(base_dir):
        full = os.path.join(base_dir, name)
        if os.path.isdir(full) and name.startswith(prefix):
            candidates.append(full)
    if not candidates:
        raise FileNotFoundError(f"No run folder found for prefix: {prefix}")
    candidates = sorted(candidates, key=lambda p: os.path.getmtime(p), reverse=True)
    return candidates[0]

def find_latest_prep_jsonl(base_dir):
    candidates = glob.glob(os.path.join(base_dir, "**", "prep_rows.jsonl"), recursive=True)
    if not candidates:
        raise FileNotFoundError("No prep_rows.jsonl found under colab_exports")
    candidates = sorted(candidates, key=lambda p: os.path.getmtime(p), reverse=True)
    return candidates[0]

prep_jsonl_path = find_latest_prep_jsonl(COLAB_EXPORTS_DIR)
prep_rows = load_jsonl(prep_jsonl_path)

for ex in prep_rows:
    ex["trec_year"] = int(ex["trec_year"])
    ex["query_id"] = str(ex["query_id"])
    ex["query"] = norm_ws(ex["query"])
    ex["labels_100"] = [int(x) for x in ex["labels_100"]]
    ex["n_candidates"] = int(ex["n_candidates"])
    ex["n_highly_relevant"] = int(ex["n_highly_relevant"])

    clean_candidates = []
    for c in ex["candidates_100"]:
        clean_candidates.append({
            "doc_id": str(c["doc_id"]),
            "score": float(c["score"]),
            "passage_text": norm_ws(c["passage_text"]),
            "source_doc_id": str(c.get("source_doc_id", ""))
        })
    ex["candidates_100"] = clean_candidates

prep_rows = sorted(prep_rows, key=lambda x: (x["trec_year"], x["query_id"]))

print("Loaded prep set from:", prep_jsonl_path)
print("Prepared queries:", len(prep_rows))
print(pd.DataFrame([{"trec_year": x["trec_year"]} for x in prep_rows]).groupby("trec_year").size().reset_index(name="n_queries").to_string(index=False))
print()

run_dirs = {}
for method, prefix in RUN_PREFIXES.items():
    run_dirs[method] = find_latest_run_dir(COLAB_IMPORTS_DIR, prefix)

print("Using run folders")
for method in METHOD_ORDER:
    if method in run_dirs:
        print(method, "->", run_dirs[method])
print()

model_results = {}

for method in ["Llama", "Qwen", "GPT", "CrossEncoder"]:
    results_path = os.path.join(run_dirs[method], "results_rows.jsonl")
    summary_path = os.path.join(run_dirs[method], "summary.json")

    rows = load_jsonl(results_path)
    summary = load_json(summary_path)

    cleaned_rows = []
    for row in rows:
        cleaned_rows.append({
            "trec_year": int(row["trec_year"]),
            "query_id": str(row["query_id"]),
            "top3": row.get("top3"),
            "top5": row.get("top5"),
            "p_at_3_rel2": row.get("p_at_3_rel2"),
            "r_at_3_rel2": row.get("r_at_3_rel2"),
            "hit_at_3_rel2": row.get("hit_at_3_rel2"),
            "ndcg_at_3": row.get("ndcg_at_3"),
            "p_at_5_rel2": row.get("p_at_5_rel2"),
            "r_at_5_rel2": row.get("r_at_5_rel2"),
            "hit_at_5_rel2": row.get("hit_at_5_rel2"),
            "ndcg_at_5": row.get("ndcg_at_5"),
            "success": row.get("success", 1)
        })

    model_results[method] = {
        "rows": pd.DataFrame(cleaned_rows),
        "summary": summary
    }

print("Loaded model result tables")
for method in ["CrossEncoder", "GPT", "Qwen", "Llama"]:
    print(method, "rows:", len(model_results[method]["rows"]))
print()

baseline_rows = []
t0_baselines = time.time()
n_total = len(prep_rows)

print("Building baseline rankings")
print()

for idx, ex in enumerate(prep_rows):
    query = ex["query"]
    labels = ex["labels_100"]
    candidates = ex["candidates_100"]

    rank_bm25, _ = bm25_rank_within(query, candidates)
    rank_mmr = mmr_rank_within_bm25rel(query, candidates, lam=MMR_LAMBDA)
    rank_rand = random_rank_within(idx, len(candidates), seed=RANDOM_SEED)

    for method, ranking in [
        ("BM25", rank_bm25),
        ("MMR", rank_mmr),
        ("Random", rank_rand),
    ]:
        baseline_rows.append({
            "method": method,
            "trec_year": int(ex["trec_year"]),
            "query_id": str(ex["query_id"]),
            "top3": ranking[:3],
            "top5": ranking[:5],
            "p_at_3_rel2": precision_at_k(labels, ranking, 3, TARGET_REL),
            "r_at_3_rel2": recall_at_k(labels, ranking, 3, TARGET_REL),
            "hit_at_3_rel2": hit_at_k(labels, ranking, 3, TARGET_REL),
            "ndcg_at_3": ndcg_at_k(labels, ranking, 3),
            "p_at_5_rel2": precision_at_k(labels, ranking, 5, TARGET_REL),
            "r_at_5_rel2": recall_at_k(labels, ranking, 5, TARGET_REL),
            "hit_at_5_rel2": hit_at_k(labels, ranking, 5, TARGET_REL),
            "ndcg_at_5": ndcg_at_k(labels, ranking, 5),
            "success": 1
        })

    done = idx + 1
    if done % BASELINE_PRINT_EVERY == 0 or done == n_total:
        elapsed = time.time() - t0_baselines
        print(
            f"{done}/{n_total} baseline queries done | "
            f"rows_built={len(baseline_rows)} | "
            f"elapsed={elapsed:.2f}s"
        )

print()

baseline_df = pd.DataFrame(baseline_rows)

for method in ["BM25", "MMR", "Random"]:
    model_results[method] = {
        "rows": baseline_df[baseline_df["method"] == method].drop(columns=["method"]).reset_index(drop=True),
        "summary": None
    }

print("Baseline result tables ready")
for method in ["BM25", "MMR", "Random"]:
    print(method, "rows:", len(model_results[method]["rows"]))
print()

print("Cell 1 finished")

Loaded prep set from: /home/baris/projects/submitted_multinews_llm_ranking_public-Revised/colab_exports/trecdl_2021_2023_k5eligible_20260412_221209/prep_rows.jsonl
Prepared queries: 125
 trec_year  n_queries
      2021         40
      2022         42
      2023         43

Using run folders
CrossEncoder -> /home/baris/projects/submitted_multinews_llm_ranking_public-Revised/colab_imports/cross_encoder_full100_20260412
GPT -> /home/baris/projects/submitted_multinews_llm_ranking_public-Revised/colab_imports/gpt_top5_20260412
Qwen -> /home/baris/projects/submitted_multinews_llm_ranking_public-Revised/colab_imports/qwen100_top5_20260412
Llama -> /home/baris/projects/submitted_multinews_llm_ranking_public-Revised/colab_imports/llama100_top5_20260412

Loaded model result tables
CrossEncoder rows: 125
GPT rows: 125
Qwen rows: 125
Llama rows: 125

Building baseline rankings

10/125 baseline queries done | rows_built=30 | elapsed=35.07s
20/125 baseline queries done | rows_built=60 | elapsed=70.

### eval

In [5]:
metric_cols = [
    "p_at_3_rel2", "r_at_3_rel2", "hit_at_3_rel2", "ndcg_at_3",
    "p_at_5_rel2", "r_at_5_rel2", "hit_at_5_rel2", "ndcg_at_5"
]

def filter_to_common(df, common_keys_set):
    key_set = set(common_keys_set)
    return df[df.apply(lambda r: (int(r["trec_year"]), str(r["query_id"])) in key_set, axis=1)].copy()

def build_symmetric_matrix(df, methods, value_col, diag_value):
    mat = pd.DataFrame(index=methods, columns=methods, dtype=float)
    for m in methods:
        mat.loc[m, m] = diag_value
    for _, row in df.iterrows():
        a = row["method_a"]
        b = row["method_b"]
        v = row[value_col]
        mat.loc[a, b] = v
        mat.loc[b, a] = v
    return mat

all_method_keys = {}
for method in METHOD_ORDER:
    df = model_results[method]["rows"]
    valid = df[df["top5"].notna()].copy()
    all_method_keys[method] = set(zip(valid["trec_year"], valid["query_id"]))

common_keys = set.intersection(*[all_method_keys[m] for m in METHOD_ORDER])
common_keys = sorted(common_keys)

print("Common comparable queries across all methods:", len(common_keys))
print()

overall_rows = []
year_rows = []

for method in METHOD_ORDER:
    df = filter_to_common(model_results[method]["rows"], common_keys)

    overall_row = {"method": method, "n_queries": len(df)}
    for col in metric_cols:
        overall_row[col] = float(df[col].dropna().mean()) if len(df[col].dropna()) else None
    if "success" in df.columns:
        overall_row["success_rate"] = float(df["success"].mean())
    else:
        overall_row["success_rate"] = 1.0
    overall_rows.append(overall_row)

    for year, sub in df.groupby("trec_year"):
        year_row = {"method": method, "trec_year": int(year), "n_queries": len(sub)}
        for col in metric_cols:
            year_row[col] = float(sub[col].dropna().mean()) if len(sub[col].dropna()) else None
        if "success" in sub.columns:
            year_row["success_rate"] = float(sub["success"].mean())
        else:
            year_row["success_rate"] = 1.0
        year_rows.append(year_row)

overall_df = pd.DataFrame(overall_rows)
overall_df["method"] = pd.Categorical(overall_df["method"], categories=METHOD_ORDER, ordered=True)
overall_df = overall_df.sort_values("method").reset_index(drop=True)

year_df = pd.DataFrame(year_rows)
year_df["method"] = pd.Categorical(year_df["method"], categories=METHOD_ORDER, ordered=True)
year_df = year_df.sort_values(["trec_year", "method"]).reset_index(drop=True)

print("Overall metrics table")
display(overall_df)

print("Per-year metrics table")
display(year_df)

top_map = {}
for method in METHOD_ORDER:
    df = filter_to_common(model_results[method]["rows"], common_keys)
    top_map[method] = {
        (int(r["trec_year"]), str(r["query_id"])): {
            "top3": list(r["top3"]),
            "top5": list(r["top5"])
        }
        for _, r in df.iterrows()
    }

pair_rows = []

for a, b in itertools.combinations(METHOD_ORDER, 2):
    jacc3 = []
    jacc5 = []
    overlap3 = []
    overlap5 = []
    exact3_list = []
    exact5_list = []
    exact3_set = []
    exact5_set = []

    for key in common_keys:
        top3_a = top_map[a][key]["top3"]
        top3_b = top_map[b][key]["top3"]
        top5_a = top_map[a][key]["top5"]
        top5_b = top_map[b][key]["top5"]

        jacc3.append(jaccard_set(top3_a, top3_b))
        jacc5.append(jaccard_set(top5_a, top5_b))
        overlap3.append(overlap_count(top3_a, top3_b, 3))
        overlap5.append(overlap_count(top5_a, top5_b, 5))
        exact3_list.append(exact_topk_list_match(top3_a, top3_b, 3))
        exact5_list.append(exact_topk_list_match(top5_a, top5_b, 5))
        exact3_set.append(exact_topk_set_match(top3_a, top3_b, 3))
        exact5_set.append(exact_topk_set_match(top5_a, top5_b, 5))

    pair_rows.append({
        "method_a": a,
        "method_b": b,
        "n_queries": len(common_keys),
        "jacc3_mean": float(np.mean(jacc3)),
        "jacc5_mean": float(np.mean(jacc5)),
        "overlap3_mean": float(np.mean(overlap3)),
        "overlap5_mean": float(np.mean(overlap5)),
        "exact_top3_list_rate": float(np.mean(exact3_list)),
        "exact_top5_list_rate": float(np.mean(exact5_list)),
        "exact_top3_set_rate": float(np.mean(exact3_set)),
        "exact_top5_set_rate": float(np.mean(exact5_set)),
    })

pairwise_df = pd.DataFrame(pair_rows).sort_values(
    ["jacc5_mean", "jacc3_mean", "exact_top5_list_rate"],
    ascending=False
).reset_index(drop=True)

print("Pairwise agreement table")
display(pairwise_df)

jacc3_matrix = build_symmetric_matrix(pairwise_df, METHOD_ORDER, "jacc3_mean", 1.0)
jacc5_matrix = build_symmetric_matrix(pairwise_df, METHOD_ORDER, "jacc5_mean", 1.0)
overlap3_matrix = build_symmetric_matrix(pairwise_df, METHOD_ORDER, "overlap3_mean", 3.0)
overlap5_matrix = build_symmetric_matrix(pairwise_df, METHOD_ORDER, "overlap5_mean", 5.0)
exact3_list_matrix = build_symmetric_matrix(pairwise_df, METHOD_ORDER, "exact_top3_list_rate", 1.0)
exact5_list_matrix = build_symmetric_matrix(pairwise_df, METHOD_ORDER, "exact_top5_list_rate", 1.0)

print("Pairwise top-3 Jaccard")
display(jacc3_matrix)

print("Pairwise top-5 Jaccard")
display(jacc5_matrix)

print("Pairwise top-3 overlap count")
display(overlap3_matrix)

print("Pairwise top-5 overlap count")
display(overlap5_matrix)

print("Pairwise exact top-3 ordered match rate")
display(exact3_list_matrix)

print("Pairwise exact top-5 ordered match rate")
display(exact5_list_matrix)

print("Cell 2 finished")

Common comparable queries across all methods: 123

Overall metrics table


,method,n_queries,p_at_3_rel2,r_at_3_rel2,hit_at_3_rel2,ndcg_at_3,p_at_5_rel2,r_at_5_rel2,hit_at_5_rel2,ndcg_at_5,success_rate
0,BM25,123,0.276423,0.059217,0.487805,0.282478,0.266667,0.098014,0.609756,0.283611,1.0
1,MMR,123,0.257453,0.055990,0.504065,0.273156,0.260163,0.094085,0.642276,0.276673,1.0
2,CrossEncoder,123,0.691057,0.184418,0.894309,0.661088,0.621138,0.269481,0.943089,0.635715,1.0
3,GPT,123,0.742547,0.199921,0.934959,0.696998,0.687805,0.303049,0.959350,0.678926,1.0
4,Qwen,123,0.512195,0.125739,0.804878,0.475560,0.409756,0.161317,0.829268,0.428115,1.0
5,Llama,123,0.468835,0.115056,0.747967,0.440011,0.463415,0.184980,0.886179,0.447276,1.0
6,Random,123,0.143631,0.027156,0.341463,0.143382,0.152846,0.050926,0.520325,0.156126,1.0


Per-year metrics table


,method,trec_year,n_queries,p_at_3_rel2,r_at_3_rel2,hit_at_3_rel2,ndcg_at_3,p_at_5_rel2,r_at_5_rel2,hit_at_5_rel2,ndcg_at_5,success_rate
0,BM25,2021,39,0.435897,0.065152,0.615385,0.380702,0.400000,0.103433,0.717949,0.376359,1.0
1,MMR,2021,39,0.401709,0.063252,0.615385,0.365885,0.379487,0.098008,0.717949,0.360362,1.0
2,CrossEncoder,2021,39,0.803419,0.168454,1.000000,0.736584,0.758974,0.248765,1.000000,0.719578,1.0
3,GPT,2021,39,0.888889,0.185140,0.974359,0.803517,0.830769,0.275207,1.000000,0.781728,1.0
4,Qwen,2021,39,0.735043,0.143937,1.000000,0.651159,0.605128,0.188053,1.000000,0.593242,1.0
5,Llama,2021,39,0.623932,0.114608,0.897436,0.538426,0.630769,0.195807,0.974359,0.557394,1.0
6,Random,2021,39,0.162393,0.025160,0.384615,0.167068,0.189744,0.050336,0.589744,0.193337,1.0
7,BM25,2022,41,0.146341,0.040246,0.341463,0.199412,0.156098,0.062691,0.414634,0.208202,1.0
8,MMR,2022,41,0.154472,0.041110,0.390244,0.202322,0.170732,0.069758,0.512195,0.213635,1.0
9,CrossEncoder,2022,41,0.691057,0.202646,0.878049,0.667348,0.619512,0.305495,0.951220,0.642079,1.0


Pairwise agreement table


...

Pairwise exact top-3 ordered match rate


,BM25,MMR,CrossEncoder,GPT,Qwen,Llama,Random
BM25,1.000000,0.634146,0.03252,0.00000,0.00000,0.00000,0.0
MMR,0.634146,1.000000,0.02439,0.00000,0.00000,0.00000,0.0
CrossEncoder,0.032520,0.024390,1.00000,0.00813,0.00813,0.00813,0.0
GPT,0.000000,0.000000,0.00813,1.00000,0.00000,0.01626,0.0
Qwen,0.000000,0.000000,0.00813,0.00000,1.00000,0.01626,0.0
Llama,0.000000,0.000000,0.00813,0.01626,0.01626,1.00000,0.0
Random,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,1.0


Pairwise exact top-5 ordered match rate


,BM25,MMR,CrossEncoder,GPT,Qwen,Llama,Random
BM25,1.000000,0.398374,0.00813,0.00000,0.0,0.00000,0.0
MMR,0.398374,1.000000,0.00000,0.00000,0.0,0.00000,0.0
CrossEncoder,0.008130,0.000000,1.00000,0.00000,0.0,0.00000,0.0
GPT,0.000000,0.000000,0.00000,1.00000,0.0,0.00813,0.0
Qwen,0.000000,0.000000,0.00000,0.00000,1.0,0.00000,0.0
Llama,0.000000,0.000000,0.00000,0.00813,0.0,1.00000,0.0
Random,0.000000,0.000000,0.00000,0.00000,0.0,0.00000,1.0


Cell 2 finished
